In [19]:
# ============================================================
# Assignment - Part 3: File I/O, APIs & Exception Handling
# Theme: Product Explorer & Error-Resilient Logger
# # ============================================================

# ============================================================
# TASK 1 - File Read & Write Basics
# ============================================================

print("=" * 55)
print("         TASK 1 - File Read & Write Basics")
print("=" * 55)

# -------------------------------
# PART A — WRITE TO FILE
# -------------------------------

file_name = "python_notes.txt"

# writing initial content
with open(file_name, "w", encoding="utf-8") as file:
    file.write("Topic 1: Variables store data. Python is dynamically typed.\n")
    file.write("Topic 2: Lists are ordered and mutable.\n")
    file.write("Topic 3: Dictionaries store key-value pairs.\n")
    file.write("Topic 4: Loops automate repetitive tasks.\n")
    file.write("Topic 5: Exception handling prevents crashes.\n")

print("File written successfully.")

# appending extra lines
with open(file_name, "a", encoding="utf-8") as file:
    file.write("Topic 6: Functions help reuse code.\n")
    file.write("Topic 7: Python supports multiple programming paradigms.\n")

print("Lines appended successfully.")


# -------------------------------
# PART B — READ FROM FILE
# -------------------------------

print("\nReading file contents:\n")

with open(file_name, "r", encoding="utf-8") as file:
    lines = file.readlines()

# print numbered lines
for i, line in enumerate(lines, start=1):
    print(f"{i}. {line.strip()}")  # strip removes \n

# total number of lines
print(f"\nTotal number of lines: {len(lines)}")


# -------------------------------
# KEYWORD SEARCH
# -------------------------------

keyword = input("\nEnter a keyword to search: ").lower()

found = False

print("\nSearch Results:")

for line in lines:
    if keyword in line.lower():  # case-insensitive match
        print(line.strip())
        found = True

if not found:
    print("No matching lines found.")



# ============================================================
# TASK 2 - API Integration
# ============================================================

print("\n" + "=" * 55)
print("           TASK 2 - API Integration")
print("=" * 55)

import requests

# Base URL
BASE_URL = "https://dummyjson.com/products"

# -------------------------------
# Step 1 — Fetch and Display Products
# -------------------------------

print("\nFetching products...\n")

response = requests.get(f"{BASE_URL}?limit=20")

# basic error check (important in real-world code)
if response.status_code != 200:
    print("Failed to fetch data from API")
    exit()

data = response.json()

products = data["products"]

# print table header
print("ID  | Title                          | Category      | Price    | Rating")
print("-" * 75)

# loop through products and print formatted data
for p in products:
    print(f"{p['id']:<3} | {p['title']:<30} | {p['category']:<13} | ${p['price']:<8} | {p['rating']}")


# -------------------------------
# Step 2 — Filter and Sort
# -------------------------------

print("\nFiltered (Rating >= 4.5) and Sorted by Price:\n")

# filter products
filtered = []
for p in products:
    if p["rating"] >= 4.5:
        filtered.append(p)

# sort by price (descending)
filtered.sort(key=lambda x: x["price"], reverse=True)

# print result
for p in filtered:
    print(f"{p['title']} - ${p['price']} (Rating: {p['rating']})")


# -------------------------------
# Step 3 — Search by Category
# -------------------------------

print("\nFetching laptops category...\n")

laptop_response = requests.get(f"{BASE_URL}/category/laptops")

if laptop_response.status_code != 200:
    print("Failed to fetch laptops data")
else:
    laptops = laptop_response.json()["products"]

    print("Laptops List:")
    for laptop in laptops:
        print(f"{laptop['title']} - ${laptop['price']}")


# -------------------------------
# Step 4 — POST Request (Simulated)
# -------------------------------

print("\nSending POST request...\n")

new_product = {
    "title": "My Custom Product",
    "price": 999,
    "category": "electronics",
    "description": "A product I created via API"
}

post_response = requests.post(f"{BASE_URL}/add", json=new_product)

# print full response
if post_response.status_code in [200, 201]:
    print("Response from server:")
    print(post_response.json())
else:
    print("POST request failed")


# ============================================================
# TASK 3 - Exception Handling
# ============================================================

print("\n" + "=" * 55)
print("          TASK 3 - Exception Handling")
print("=" * 55)

# ---- Part A: Guarded Calculator ----

print("\nPart A -- safe_divide function:\n")

# safe division with basic error handling
def safe_divide(a, b):
    try:
        return a / b

    except ZeroDivisionError:
        return "Error: Cannot divide by zero"

    except TypeError:
        return "Error: Invalid input types"


# test cases
print("Part A — Calculator:")
print(safe_divide(10, 2))
print(safe_divide(10, 0))
print(safe_divide("ten", 2))

# ---- Part B: Guarded File Reader ----

def read_file_safe(filename):
    try:
        # try opening and reading file
        with open(filename, "r", encoding="utf-8") as file:
            content = file.read()
            return content

    except FileNotFoundError:
        print(f"Error: File '{filename}' not found.")

    finally:
        # this runs no matter what
        print("File operation attempt complete.")


print("\nPart B — File Reader:")

# should work
print(read_file_safe("python_notes.txt"))

# should fail gracefully
print(read_file_safe("ghost_file.txt"))


# ---- Part C: Robust API Calls ----

import requests

BASE_URL = "https://dummyjson.com/products"

print("\nPart C — API Handling:")

try:
    # adding timeout to simulate real-world behavior
    response = requests.get(f"{BASE_URL}?limit=5", timeout=5)

    if response.status_code == 200:
        data = response.json()
        print("Products fetched successfully.")
    else:
        print("API returned an error:", response.status_code)

# network-level errors
except requests.exceptions.ConnectionError:
    print("Connection failed. Please check your internet.")

except requests.exceptions.Timeout:
    print("Request timed out. Try again later.")

# catch anything unexpected
except Exception as e:
    print("Unexpected error:", e)

# ---- Part D: Input Validation Loop ----

print("\nPart D — Product Lookup:")

while True:
    user_input = input("Enter product ID (1–100) or 'quit': ")

    # exit condition
    if user_input.lower() == "quit":
        print("Exiting...")
        break

    # validate input is number
    if not user_input.isdigit():
        print("Invalid input. Please enter a number.")
        continue

    product_id = int(user_input)

    # validate range
    if product_id < 1 or product_id > 100:
        print("Please enter a number between 1 and 100.")
        continue

    # API call with error handling
    try:
        response = requests.get(f"{BASE_URL}/{product_id}", timeout=5)

        if response.status_code == 200:
            product = response.json()
            print(f"{product['title']} - ₹{product['price']}")

        elif response.status_code == 404:
            print("Product not found.")

        else:
            print("Something went wrong:", response.status_code)

    except requests.exceptions.ConnectionError:
        print("Connection failed. Please check your internet.")

    except requests.exceptions.Timeout:
        print("Request timed out. Try again later.")

    except Exception as e:
        print("Unexpected error:", e)

# ============================================================
# Task 4 — Logging to File
# ============================================================

import requests
from datetime import datetime

log_file = "error_log.txt"

# -------------------------------
# Helper function to write logs
# -------------------------------
def log_error(function_name, error_type, message):

    # get current timestamp
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    # format log entry
    log_entry = f"[{timestamp}] ERROR in {function_name}: {error_type} — {message}\n"

    # append to file (important: 'a' mode)
    with open(log_file, "a", encoding="utf-8") as file:
        file.write(log_entry)


# -------------------------------
# 1. Trigger ConnectionError
# -------------------------------
print("Triggering ConnectionError...\n")

try:
    requests.get("https://this-host-does-not-exist-xyz.com/api", timeout=5)

except requests.exceptions.ConnectionError as e:
    print("Connection error caught.")
    log_error("fetch_products", "ConnectionError", str(e))


# -------------------------------
# 2. Trigger HTTP error (404)
# -------------------------------
print("Triggering HTTP 404 error...\n")

try:
    response = requests.get("https://dummyjson.com/products/999", timeout=5)

    # 404 is NOT an exception → handle manually
    if response.status_code != 200:
        print("HTTP error detected.")

        log_error(
            "lookup_product",
            "HTTPError",
            f"{response.status_code} Not Found for product ID 999"
        )

except requests.exceptions.ConnectionError as e:
    log_error("lookup_product", "ConnectionError", str(e))

except requests.exceptions.Timeout:
    log_error("lookup_product", "Timeout", "Request timed out")

except Exception as e:
    log_error("lookup_product", "UnknownError", str(e))


# -------------------------------
# 3. Read and display log file
# -------------------------------
print("\nError Log Contents:\n")

try:
    with open(log_file, "r", encoding="utf-8") as file:
        print(file.read())

except FileNotFoundError:
    print("Log file not found.")

         TASK 1 - File Read & Write Basics
File written successfully.
Lines appended successfully.

Reading file contents:

1. Topic 1: Variables store data. Python is dynamically typed.
2. Topic 2: Lists are ordered and mutable.
3. Topic 3: Dictionaries store key-value pairs.
4. Topic 4: Loops automate repetitive tasks.
5. Topic 5: Exception handling prevents crashes.
6. Topic 6: Functions help reuse code.
7. Topic 7: Python supports multiple programming paradigms.

Total number of lines: 7

Enter a keyword to search: variables

Search Results:
Topic 1: Variables store data. Python is dynamically typed.

           TASK 2 - API Integration

Fetching products...

ID  | Title                          | Category      | Price    | Rating
---------------------------------------------------------------------------
1   | Essence Mascara Lash Princess  | beauty        | $9.99     | 2.56
2   | Eyeshadow Palette with Mirror  | beauty        | $19.99    | 2.86
3   | Powder Canister              